These notebooks apply ArchVelo regression to human peripheral blood mononuclear cells (PBMC). The preprocessed data (from https://www.10xgenomics.com/datasets/pbmc-from-a-healthy-donor-granulocytes-removed-through-cell-sorting-10-k-1-standard-2-0-0) and other necessary files can be downloaded at https://doi.org/10.6084/m9.figshare.31909822

In [1]:
import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import ArchVelo as av

In [2]:
import os
outdir = 'test_raw/'
os.makedirs(outdir, exist_ok = True)

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
train_atac= sc.read_h5ad('data/PBMC/train_atac.h5ad')
test_atac= sc.read_h5ad('data/PBMC/test_atac.h5ad')

In [6]:
#splitting like this because cells are already shuffled. Otherwise - shuffle
num_cells = train_atac.shape[0]
l = [0]*int(num_cells/3)+[1]*int(num_cells/3)+[2]*(train_atac.shape[0] - 2*int(num_cells/3))

In [9]:
rna = sc.read_h5ad('data/PBMC/pbmc_processed.h5ad')

In [10]:
peaks_to_genes = pd.read_csv('data/PBMC/nearest_genes.pbmc.csv', index_col = [0])
peaks_to_genes.set_index('name', inplace = True)

In [11]:
gene_counts = peaks_to_genes.loc[train_atac.var.index,:]['gene'].value_counts()
all_genes = gene_counts.index.values

In [12]:
train_ind_orig = [np.where(rna.obs.index == '-'.join(x.split('-')[1:]))[0][0] for x in train_atac.obs.index]
test_ind_orig = [np.where(rna.obs.index == '-'.join(x.split('-')[1:]))[0][0] for x in test_atac.obs.index]


In [13]:
import pickle

In [14]:
import time

In [17]:
all_genes = gene_counts.index.values
XC_train = None
XC_test = None
S = None
XC_train_folds = {}
XC_test_folds = {}
S_folds = {}
for fld in [0,1,2]:
    XC_train_folds[fld] = None
    XC_test_folds[fld] = None
    S_folds[fld] = None
        
results_raw = av.atac_regression_with_components(
                               genes = all_genes,
                               XC_train = XC_train, 
                               XC_test = XC_test,
                               S = S, 
                               XC_train_folds = XC_train_folds, 
                               XC_test_folds = XC_test_folds,
                               S_folds = S_folds, 
                               cross_validate = True,
                               l = l,
                               rna_data = rna,
                               train_df = train_atac, 
                               test_df = test_atac,
                               train_ind_orig = train_ind_orig,
                               test_ind_orig = test_ind_orig,
                               peaks_to_genes = peaks_to_genes,
                               feature_type = 'raw',
                               rna_layer = 'log1p',
                               metr = 'pearson',
                               atac_layer = 'pearson',
                               verbose = True)  
f = open('test_raw/rna_from_atac_pearson_corrs_CV.p', 'wb')
pickle.dump(results_raw, f)
f.close()

FOXP1
Best regularization:  10481.13134154683
Validation accuracy:  0.3119350545497155
Test accuracy:  0.2850162491684723
RAD51B
Best regularization:  10481.13134154683
Validation accuracy:  -0.00324719494737727
Test accuracy:  0.0023204937973171475
SEPTIN9
Best regularization:  100000.0
Validation accuracy:  0.024471883569651454
Test accuracy:  0.05851982641504515
UBAC2
Best regularization:  100000.0
Validation accuracy:  0.025839978014456904
Test accuracy:  -0.010282619038303565
RUNX1
Best regularization:  1e-07
Validation accuracy:  0.07100334461622354
Test accuracy:  0.036632050887371016
SEMA4D
Best regularization:  100000.0
Validation accuracy:  0.1311205253195075
Test accuracy:  0.07214740962730638
HDAC4
Best regularization:  100000.0
Validation accuracy:  0.1953673612774106
Test accuracy:  0.21021032778342122
PRKCH
Best regularization:  5963.623316594637
Validation accuracy:  0.5554248442930522
Test accuracy:  0.5497802057284563
ANKRD44
Best regularization:  100000.0
Validation 